# <font color=#0099CC>**Diseño de Algoritmos de Inversión y Backtesting — Momentum MSCI**</font>

### **Fecha: 29/02/2026 | Autor: Javier Fernández Guerra**



# <font color=#0099CC>**Notebook 1: Carga de Datos**</font>

En este notebook se cargan los datos necesarios para el backtesting: el histórico del S&P 500 (precios diarios y pertenencia al índice) y el benchmark SPY (S&P 500 ETF Trust). Todo el pipeline posterior depende de estos datos.

In [1]:
# Librerías
import sys
from pathlib import Path

import pandas as pd
import yfinance as yf

# Permite importar utilidades compartidas desde la raíz del proyecto
project_root = Path.cwd().resolve()
if not (project_root / 'notebook_utils.py').exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from notebook_utils import configure_notebook_display, get_project_paths

# Configuración de visualización (consistente en todos los notebooks)
configure_notebook_display(
    max_columns=None,
    max_rows=100,
    width=None,
    max_colwidth=None,
)

# Rutas del proyecto centralizadas en una clase reutilizable
PATHS = get_project_paths(project_root)
DATASETS_DIR = PATHS.datasets

> <u>Nota</u>: Este notebook usa utilidades compartidas del módulo `notebook_utils.py` para centralizar rutas y configuración de visualización. Con ello se mejora la modularidad, la reutilización del código y la consistencia de estilo (PEP 8) entre notebooks.

## <font color=#0099CC>**1. CARGA DEL HISTÓRICO S&P 500**</font>

### <font color=#336699>**1.1. Lectura del archivo Parquet**</font>

Se carga el fichero `sp500_history.parquet`, que incluye, por fecha y símbolo, información sobre todas las empresas que han formado parte del índice S&P 500 (open, high, low, close, volume y flag `in_sp500`, entre otros). Este histórico es la base para construir el universo de inversión y los precios diarios en los notebooks siguientes.

In [2]:
# Cargar el archivo parquet desde la carpeta datasets
sp500_path = DATASETS_DIR / 'sp500_history.parquet'
df_sp500_pre = pd.read_parquet(sp500_path)
df_sp500 = df_sp500_pre.copy()

# Mostrar parquet
print(f'> Parquet de datos históricos S&P 500 cargado desde {sp500_path}')
display(df_sp500.head())

> Parquet de datos históricos S&P 500 cargado desde C:\Users\Javi\Desktop\backtesting\datasets\sp500_history.parquet


,date,symbol,assetid,security_name,sector,industry,subsector,in_sp500,open,high,low,close,volume,unadjusted_close
0,1999-11-18,A,131684,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,0,27.188307,29.877260,23.901808,25.545057,74862288.0,42.7500
1,1999-11-19,A,131684,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,0,25.657097,25.694445,23.789768,24.349968,18236110.0,40.7500
2,1999-11-22,A,131684,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,0,24.686087,26.067909,23.939156,26.067909,7874048.5,43.6250
3,1999-11-23,A,131684,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,0,25.395672,26.067909,24.051195,24.051195,7153099.0,40.2500
4,1999-11-24,A,131684,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,0,23.976501,25.059551,23.901808,24.536699,5797720.5,41.0625


> <u>Comentario</u>: El DataFrame muestra una fila por día y por símbolo, con precios ajustados y el indicador de pertenencia al índice. No se requiere limpieza adicional en esta etapa (se hace en el Notebook 2). La cobertura temporal y de símbolos es la que utilizará el Notebook 2 para preparar retornos y el Notebook 3 para el universo de los últimos 13 meses.

## <font color=#0099CC>**2. BENCHMARK SPY**</font>

### <font color=#336699>**2.1. Carga o descarga del SPY**</font>

Se utiliza el SPY (S&P 500 ETF Trust) como benchmark. Utilizamos precios ajustados (`auto_adjust=True`) para mantener consistencia con los precios ajustados del universo de acciones. El fichero `spy_benchmark.parquet` se regenera desde Yahoo Finance y se guarda en `../datasets/`.

In [3]:
# Parámetros
parquet_path = DATASETS_DIR / 'spy_benchmark.parquet'
start_date = '1999-11-18'
end_date = '2026-01-30'

# Benchmark SPY SIEMPRE ajustado (total return)
spy = yf.download(
    'SPY',
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=False,
)

# Sobrescribir parquet para evitar mezclar versiones antiguas/no ajustadas
spy.to_parquet(parquet_path)
print(f"> Benchmark SPY ajustado descargado y guardado en: {parquet_path}")

# Mostrar primeras filas del DataFrame
display(spy.head())

> Benchmark SPY ajustado descargado y guardado en: C:\Users\Javi\Desktop\backtesting\datasets\spy_benchmark.parquet


Price,Close,High,Low,Open,Volume
Ticker,SPY,SPY,SPY,SPY,SPY
Date,,,,,
1999-11-18,89.625366,89.861016,88.996967,89.507541,4491000
1999-11-19,89.546768,89.841330,89.232569,89.487856,4832100
1999-11-22,89.527184,89.861020,88.918422,89.507546,4155400
1999-11-23,88.741676,89.762824,88.211465,89.762824,5918000
1999-11-24,89.212967,89.507529,87.975807,88.447106,4459700


> <u>Comentario</u>: Con el SPY cargado (o descargado) se dispone del benchmark contra el que se comparará la estrategia Momentum en el Notebook 5. La serie de precios se utilizará en el Notebook 4 para construir la curva de equity del benchmark y en el Notebook 2 si se necesita alinear fechas con el histórico del S&P 500.